# 09 — Pod all-reduce: per-step time, communication fraction, scaling efficiency

**Learning goal.** Reason quantitatively about strong/weak scaling on a TPU pod. Show why doubling chips does **not** halve step time, and why for small payloads adding chips can actually hurt.

**What you'll observe.**
- Per-step time as `n_chips` grows from 1 to 32: compute drops linearly, comm grows sub-linearly, total bottoms out and then rises.
- The communication fraction of step time vs `n_chips`.
- The scaling efficiency curve: `(t_1 / t_N) / N`.

**Knobs to tune.**
- `compute_per_chip_s_1chip` — baseline per-chip compute work at 1 chip. Pure data-parallel divides this by `N`.
- `payload_bytes` — gradient payload sent through all-reduce each step.
- `tpu_version` — picks the ICI bandwidth.

**Failure modes to watch.**
- For very small payloads the latency term dominates — more chips strictly worsens step time.
- "Scaling efficiency" must be defined carefully: use **strong scaling** (fixed problem size, more chips) for these plots, not weak scaling.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## Sweep n_chips = 1..32

In [ ]:
from cloud_tpu_lab.src.sharding.all_reduce import all_reduce_time
from cloud_tpu_lab.src.tpu_versions.cloud_tpu_catalog import get_spec

spec = get_spec("v5e")
ici_bw_bytes_s = spec.ici_bandwidth_gbps * 1e9
print("Using", spec.code_name, "ICI bw:", spec.ici_bandwidth_gbps, "GB/s")

# Baseline compute (s) at 1 chip — small toy number; strong-scales as 1/N
compute_per_chip_s_1chip = 0.010
# Gradient payload to all-reduce each step (bytes)
payload_bytes = 4 * 1024 * 1024  # 4 MiB

chip_counts = [1, 2, 4, 8, 16, 32]
rows = []
for n in chip_counts:
    compute_s = compute_per_chip_s_1chip / n   # strong scaling
    coll = all_reduce_time(payload_bytes, n, ici_bw_bytes_s)
    comm_s = coll.sim_duration_s
    total_s = compute_s + comm_s
    rows.append({"n": n, "compute_s": compute_s, "comm_s": comm_s, "total_s": total_s})

for r in rows:
    print(f"n={r['n']:>3}  compute={r['compute_s']*1000:6.3f} ms  "
          f"comm={r['comm_s']*1000:6.3f} ms  total={r['total_s']*1000:6.3f} ms")

## Plot 1: per-step time breakdown

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ns = [r["n"] for r in rows]
comp = [r["compute_s"] * 1000 for r in rows]
comm = [r["comm_s"] * 1000 for r in rows]
total = [r["total_s"] * 1000 for r in rows]

fig = plt.figure(figsize=(7, 4))
plt.plot(ns, comp, marker="o", label="compute (strong scaling)")
plt.plot(ns, comm, marker="s", label="all-reduce comm")
plt.plot(ns, total, marker="^", label="total step", linewidth=2)
plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("n_chips")
plt.ylabel("time per step (ms, log)")
plt.title(f"Strong scaling on {spec.code_name}, payload={payload_bytes/1024/1024:.1f} MiB")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb09_strong_scaling.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Plot 2: communication fraction

In [ ]:
comm_fraction = [r["comm_s"] / r["total_s"] for r in rows]

fig = plt.figure(figsize=(7, 4))
plt.plot(ns, comm_fraction, marker="o")
plt.xscale("log", base=2)
plt.xlabel("n_chips")
plt.ylabel("fraction of step in collectives")
plt.title("Communication fraction grows with chip count")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb09_comm_fraction.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Plot 3: scaling efficiency

Strong scaling efficiency at `N` chips =  `(t_1 / t_N) / N`.

A value of 1.0 means linear scaling (perfect). 0.5 means you needed 2x as many chips to halve step time.

In [ ]:
t1 = rows[0]["total_s"]
efficiency = [(t1 / r["total_s"]) / r["n"] for r in rows]

fig = plt.figure(figsize=(7, 4))
plt.plot(ns, efficiency, marker="o")
plt.axhline(1.0, color="green", linestyle="--", label="ideal (linear scaling)")
plt.axhline(0.5, color="red", linestyle=":", label="half-efficient")
plt.xscale("log", base=2)
plt.xlabel("n_chips")
plt.ylabel("strong scaling efficiency")
plt.title("Scaling efficiency vs chip count")
plt.ylim(0, 1.1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb09_scaling_efficiency.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Why does adding chips hurt for small payloads?

With strong scaling, compute time = `T / N`. Communication time with ring all-reduce is roughly:

  `t_comm ~ alpha(N-1) + beta(2(N-1)/N) * B`

For very small `B`, the **latency** term `alpha(N-1)` dominates and grows linearly with `N`. The compute saving is only `T(1 - 1/N)`. Once `alpha(N-1) > T(1 - 1/N)`, more chips makes step time **worse**.

Concretely, if your gradient payload per all-reduce is a few KiB, you're already in the latency-dominated regime at small pod sizes. In practice this is why people either:

- Batch many small all-reduces into one (gradient bucketing).
- Run them async overlapped with backward pass.

## Exercises

1. Reduce `payload_bytes` to 4096 (4 KiB). At what `N` does step time start increasing? Replot.
2. Switch to `get_spec("v5p")` and replot. Where does the latency floor move?
3. Plot **weak** scaling: keep per-chip work constant (do not divide compute by `N`). What does efficiency look like now?
4. Add a constant `host_overhead_s` of 0.5 ms per step. How does the efficiency curve change?
5. Estimate: at what `payload_bytes` is adding chips never harmful for the spec above? (Hint: find the crossover algebraically and verify on the plot.)